# 03. 결과 검토·배포 묶음 내보내기 (Review & Export)

이 노트북은 `pyrolysis.analysis` 와 `pyrolysis.artifacts` 의 공통 함수를 호출한다.  
상세 연구 기록은 `runs/<run_id>/` 에 저장한다.  
합성 설정 점검은 CLI `python -m pyrolysis.cli report --demo` / `export --demo` 과 **같은 공통 함수**를 호출한다.

**커밋 전: Kernel → Restart & Clear Output 후 저장.** 실제 연구 셀 출력을 저장하지 않는다.

In [ ]:
from pathlib import Path

from sklearn.model_selection import train_test_split

from pyrolysis.analysis import (
    compute_variable_importance,
    save_candidate_comparison,
    save_pred_vs_actual,
    save_variable_importance,
)
from pyrolysis.config import load_problem
from pyrolysis.training import make_demo_data, run_demo_training

problem_path = Path('configs/demo.yaml')
spec = load_problem(problem_path)
result = run_demo_training(spec, Path('runs'))
run_dir = Path(result['run_dir'])
report_dir = run_dir / 'report'
target_id = result['target_id']

In [ ]:
# 예측·실측 비교, 후보 비교표, 변수 기여도 (명시적 permutation 방법).
# 실제 자료에 적용할 분석 방법은 작업 27 에서 확정한다.
X, y = make_demo_data(spec.inputs, spec.targets, n_samples=100, random_seed=42)
X_train, X_eval, y_train, y_eval = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
pipeline = result['pipeline']
y_pred = pipeline.predict(X_eval)

pva = save_pred_vs_actual(y_eval, y_pred, report_dir, target_id, unit=result.get('unit'))
cmp = save_candidate_comparison(
    [{'candidate': result['model'], 'score': result['mae'], 'ran': True, 'reason': ''}],
    report_dir, target_id, metric_name='mae', metric_unit=result.get('unit'),
)
imp = compute_variable_importance(pipeline, X_eval, y_eval, method='permutation', n_repeats=5, random_seed=42)
jpath, ppath = save_variable_importance(imp, report_dir, target_id)

print('보고서 위치:', report_dir)
print('***', result['disclaimer'], '***')
print(imp['disclaimer'])

In [ ]:
# demo 묶음 저장·재읽기 점검 (작업 4). 연구용 완전 묶음(manifest·환경 검사)은
# 평가 완료 대상이 필요하므로 작업 28/29 에서 실제 설정으로 export_bundle 로 생성한다.
from pyrolysis.artifacts import DEMO_BUNDLE_MARKER, load_bundle, save_demo_bundle

target_spec = spec.targets[0]
bundle_dir = save_demo_bundle(
    pipeline=pipeline,
    input_specs=[s for s in spec.inputs if s.id in target_spec.input_ids],
    target_spec=target_spec,
    output_dir=Path('deployment/bundles'),
    bundle_id='demo_notebook_export',
    disclaimer=result['disclaimer'],
)
reloaded = load_bundle(bundle_dir)
print('묶음 위치:', bundle_dir)
print('묶음 유형:', reloaded['contract']['bundle_type'])
print('demo 묶음은 연구용 활성화 경로가 거부한다:', reloaded['contract']['bundle_type'] == DEMO_BUNDLE_MARKER)

같은 보고서·내보내기를 CLI 로 실행:

```powershell
python -m pyrolysis.cli report --demo --output runs
python -m pyrolysis.cli export --demo --bundle-dir deployment/bundles --bundle-id demo_cli_export
```